# Workshop Notebook 2: Model Diagnostics and Architecture Augmentation

**CIROH Developer's Conference 2026 | Foundations of Machine Learning**

---

## Overview

A model that achieves NSE > 0.7 on average is promising — but the *mean* hides a lot. In this notebook we will:

1. **Diagnose** — where and when does the baseline LSTM fail?
2. **Augment with static attributes** — give the model information about basin characteristics
3. **Deepen the architecture** — add layers and explore structural modifications
4. **Compare** — side-by-side evaluation of all variants

## When should you modify a model's architecture?

Signs that structural augmentation may help:
- Systematic bias (model always over- or under-predicts in certain conditions)
- Consistent failure across seasons or flow regimes (e.g., always misses peaks)
- Large inter-basin NSE variance that correlates with a basin attribute

Signs that architecture is *not* the bottleneck:
- Training loss is much lower than validation loss -> regularization problem first
- Some basins have very few observations -> data quality problem


In [ ]:
# Preliminaries -- imports, device setup, and workshop flags.

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cpu")

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(Path(".")))

# ------------------------------------------------------------------------------
# Workshop flags
# ------------------------------------------------------------------------------
# (1) TRAIN_FROM_SCRATCH = False to load pre-trained weights instead of
#   training — useful during the workshop to save time.
# (2) SAVE_WEIGHTS = True to save model weights after training so they can be
#   reused later.

TRAIN_FROM_SCRATCH = False  # False = load pre-saved weights
SAVE_WEIGHTS       = False  # True = save weights after each training run

WEIGHTS_DIR = ROOT / "weights"
if SAVE_WEIGHTS:
    WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------

from camels_loader import CamelsSubsetLoader, FORCING_NAMES, ATTRIBUTE_NAMES
from utils import (
    StreamflowDataset,
    masked_mse_loss,
    nse_score,
    count_params,
    train,
    predict_full_timeseries,
    make_attr_dataloaders,
)

from plots import (
    plot_seasonal_bias_and_error_magnitude,
    plot_flow_duration_curves,
    plot_nse_vs_attributes,
    plot_model_comparison,
    plot_validation_loss_comparison,
)

print("Imports OK.")

In [ ]:
# Same preprocessing pipeline as Notebook 1 — run this cell to set up shared state.

loader = CamelsSubsetLoader(
    pickle_path=str(DATA_DIR / "camels_daymetv2_subset"),
    gage_id_path=str(DATA_DIR / "gage_id_subset.npy"),
)

TRAIN_END = "1999-09-30"
VAL_END = "2008-09-30"
SEQ_LEN = 365
STRIDE = 7

dates_idx = pd.DatetimeIndex(loader.dates)
train_mask = dates_idx <= pd.Timestamp(TRAIN_END)
val_mask = (dates_idx > pd.Timestamp(TRAIN_END)) & (dates_idx <= pd.Timestamp(VAL_END))
test_mask = dates_idx > pd.Timestamp(VAL_END)
dates_test = loader.dates[test_mask]

x_train = loader.forcings[train_mask]
x_val = loader.forcings[val_mask]
x_test = loader.forcings[test_mask]
y_train = loader.target[train_mask]
y_val = loader.target[val_mask]
y_test = loader.target[test_mask]

x_mean = x_train.mean(axis=(0, 1), keepdims=True)
x_std = x_train.std(axis=(0, 1), keepdims=True) + 1e-8
x_train_norm = (x_train - x_mean) / x_std
x_val_norm = (x_val - x_mean) / x_std
x_test_norm = (x_test - x_mean) / x_std

y_log_mean = float(np.nanmean(np.log1p(np.clip(y_train, 0, None))))
y_log_std = float(np.nanstd(np.log1p(np.clip(y_train, 0, None))[~np.isnan(y_train)])) + 1e-8

def normalize_target(y):
    return (np.log1p(np.clip(y, 0, None)) - y_log_mean) / y_log_std

def denormalize_target(y_norm):
    return np.expm1(y_norm * y_log_std + y_log_mean)

y_train_norm = normalize_target(y_train)
y_val_norm = normalize_target(y_val)
y_test_norm = normalize_target(y_test)
obs_cfs_test = loader.target[test_mask, :, 0]
N_FEATURES = len(FORCING_NAMES)

train_loader = DataLoader(
    StreamflowDataset(x_train_norm, y_train_norm, SEQ_LEN, STRIDE),
    batch_size=128, shuffle=True, drop_last=True,
)
val_loader = DataLoader(
    StreamflowDataset(x_val_norm, y_val_norm, SEQ_LEN, STRIDE),
    batch_size=128,
)

print(loader)

In [ ]:
# Train the baseline model to use as a reference point.
# This is the same LstmModel from Notebook 1 — same architecture, same idea.

class LstmModel(nn.Module):
    """Sequence-to-sequence LSTM for daily streamflow prediction.

    Every PyTorch model follows the same two-step pattern:
      1. Define layers in __init__
      2. Connect them in forward()

    This model is a four-layer pipeline:
        input_proj  →  lstm  →  dropout  →  output_proj
    """

    def __init__(self, n_features, hidden_size=64, n_layers=1, dropout=0.0):
        super().__init__()

        # Layer 1 — Input projection (Linear)
        # Expands each day's n_features climate variables into a
        # hidden_size-dimensional vector the LSTM can work with.
        self.input_proj = nn.Linear(n_features, hidden_size)

        # Layer 2 — LSTM (Long Short-Term Memory)
        # Processes the sequence one day at a time while carrying a hidden state
        # (memory) forward.
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=n_layers,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=True,
        )

        # Layer 3 — Dropout
        # Randomly zeros some activations during training to prevent overfitting.
        self.dropout = nn.Dropout(dropout)

        # Layer 4 — Output projection (Linear)
        # Squeezes hidden_size dimensions down to 1 number per day: predicted streamflow.
        self.output_proj = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x                (batch, seq_len, n_features)
        x = torch.relu(self.input_proj(x))
        # x                (batch, seq_len, hidden_size)
        x, _ = self.lstm(x)
        # x                (batch, seq_len, hidden_size)
        x = self.output_proj(self.dropout(x))
        # x                (batch, seq_len, 1)
        return x.squeeze(-1)
        # return           (batch, seq_len)


baseline = LstmModel(N_FEATURES, hidden_size=64, dropout=0.4).to(device)
print(f"Baseline parameters: {count_params(baseline):,}")

tl_base, vl_base = train(
    model=baseline,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=30,
    weights_path=WEIGHTS_DIR / "nb2_baseline.pt",
    train_from_scratch=TRAIN_FROM_SCRATCH,
    save_weights=SAVE_WEIGHTS,
)

pred_base_cfs = denormalize_target(predict_full_timeseries(baseline, x_test_norm, SEQ_LEN))
nse_base = {gid: nse_score(pred_base_cfs[:, i], obs_cfs_test[:, i])
            for i, gid in enumerate(loader.gage_ids)}

print("\nBaseline NSE (test):")
for gid, nse in nse_base.items():
    print(f"  {gid}: {nse:.3f}")
print(f"  Mean: {np.nanmean(list(nse_base.values())):.3f}")


---
## 1. Model Diagnostics

Before making any architectural changes, we should understand *where* and *why* the model struggles. Good diagnostics prevent us from making changes that don't address the actual problem.

We'll examine:
- **Residual patterns** — does the error have structure (seasonal, flow-magnitude)?
- **Flow duration curves** — does the model capture the full distribution of flow?
- **Basin-level correlation** — do NSE scores correlate with basin attributes?


In [ ]:
# What to look for: systematic over/under-prediction in specific months
# suggests a missing seasonal process — snowmelt, leaf-out, summer ET peaks.
plot_seasonal_bias_and_error_magnitude(dates_test, obs_cfs_test, pred_base_cfs)

In [ ]:
# What to look for: does the model track high flows (left side) and low flows (right side)?
# A gap at the left means the model misses flood peaks; at the right, it misses droughts.
plot_flow_duration_curves(obs_cfs_test, pred_base_cfs, loader.gage_ids, nse_base)

In [ ]:
# What to look for: a strong correlation (|r| > 0.5) between an attribute and NSE
# points to information the baseline model is missing.
attr_keys = ["area_gages2", "frac_forest", "aridity", "elev_mean", "soil_porosity", "frac_snow"]
plot_nse_vs_attributes(loader.attributes, ATTRIBUTE_NAMES, loader.gage_ids, nse_base, attr_keys)

**Discussion questions:**
- Is there a seasonal pattern in the errors? What physical process might cause this?
- Does the model over- or under-predict high-flow events (left end of the FDC)?
- Which basin attribute most strongly correlates with poor NSE? What might be missing?

One common finding: models trained without **static basin attributes** struggle most on atypical basins (high aridity, high snow fraction, large area). This motivates the next section.



---
## 2. Augmentation: Static Basin Attributes

The baseline model only sees **dynamic** climate forcings — it has no knowledge of
basin physical characteristics. Two basins receiving identical precipitation can respond
very differently depending on soil permeability, slope, and vegetation.

By embedding static attributes into the model, we allow it to *condition* its
predictions on those characteristics.

### Architecture modification

```
Baseline:
  x_dynamic  (batch, seq_len, 6) -> LSTM -> prediction

With attributes:
  x_dynamic (batch, seq_len, 6)                                
  x_static (batch, 35) -> MLP -> repeat across time | cat -> LSTM  -> prediction
```


In [ ]:
# Normalize static attributes
attrs_raw = loader.attributes.astype(np.float32) # (10, 35)
attrs_mean = attrs_raw.mean(axis=0, keepdims=True)
attrs_std = attrs_raw.std(axis=0, keepdims=True) + 1e-8
attrs_norm = (attrs_raw - attrs_mean) / attrs_std

N_ATTRS = attrs_norm.shape[1]
print(f"Static attributes: {N_ATTRS} features per basin")

# Dataset/DataLoader plumbing is moved to utils.py so this cell stays focused
# on the concept: adding static basin attributes as a second input.
train_loader_a, val_loader_a = make_attr_dataloaders(
    x_train_norm, y_train_norm,
    x_val_norm, y_val_norm,
    attrs_norm,
    SEQ_LEN, STRIDE,
    batch_size=128,
)

x_dyn, x_stat, y_b = next(iter(train_loader_a))
print(f"x_dynamic: {tuple(x_dyn.shape)} (batch, seq_len, forcings)")
print(f"x_static: {tuple(x_stat.shape)} (batch, n_attrs)")
print(f"y: {tuple(y_b.shape)} (batch, seq_len)")


In [ ]:
class LstmWithAttrs(nn.Module):
    """LSTM conditioned on static basin attributes.

    The key idea: encode each basin's static attributes (soil type, area, etc.)
    into a short embedding vector, then repeat it across every timestep and
    concatenate it with the dynamic forcings. The LSTM then sees both
    "what the weather is doing today" and "what kind of basin this is" at
    every step.

    Parameters
    ----------
    n_features: number of dynamic forcing variables
    n_attrs: number of static basin attributes
    hidden_size: LSTM hidden units
    attr_embed: size of the attribute embedding before concatenation
    dropout: dropout probability
    """

    def __init__(self, n_features: int, n_attrs: int,
                 hidden_size: int = 64, attr_embed: int = 32,
                 dropout: float = 0.0):
        super().__init__()

        # Small MLP that compresses 35 basin attributes → attr_embed dimensions
        self.attr_encoder = nn.Sequential(
            nn.Linear(n_attrs, attr_embed),
            nn.Tanh(),
        )
        # Input projection now sees forcings + attribute embedding side by side
        self.input_proj = nn.Linear(n_features + attr_embed, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Linear(hidden_size, 1)

    def forward(self, x_dyn: torch.Tensor, x_stat: torch.Tensor) -> torch.Tensor:
        # x_dyn       (batch, seq_len, n_features)     Shape: (128, 365,  6)
        # x_stat      (batch, n_attrs)                 --- (128,  35)

        # Compress basin attributes into a compact embedding
        attr_emb = self.attr_encoder(x_stat)
        # attr_emb    (batch, attr_embed)              --- (128,  32)

        # Repeat the same embedding at every timestep so it lines up with x_dyn
        attr_emb = attr_emb.unsqueeze(1).expand(-1, x_dyn.size(1), -1)
        # attr_emb    (batch, seq_len, attr_embed)     --- (128, 365, 32)
        # Concatenate forcings + attributes, then project to hidden size
        x = torch.relu(self.input_proj(torch.cat([x_dyn, attr_emb], dim=-1)))
        # x           (batch, seq_len, hidden_size)    --- (128, 365, 64)

        out, _ = self.lstm(x)
        return self.output_proj(self.dropout(out)).squeeze(-1)
        # return      (batch, seq_len)                 --- (128, 365)

In [ ]:
# Train the model and get results: same training loop as before, just with the new model and dataloaders.

model_attrs = LstmWithAttrs(
    n_features=N_FEATURES, n_attrs=N_ATTRS,
    hidden_size=64, attr_embed=32, dropout=0.4
).to(device)

print(f"LstmWithAttrs parameters: {count_params(model_attrs):,}")

tl_attrs, vl_attrs = train(
    model=model_attrs,
    train_loader=train_loader_a,
    val_loader=val_loader_a,
    n_epochs=30,
    weights_path=WEIGHTS_DIR / "nb2_attrs.pt",
    train_from_scratch=TRAIN_FROM_SCRATCH,
    save_weights=SAVE_WEIGHTS,
)

pred_attrs_cfs = denormalize_target(
    predict_full_timeseries(model_attrs, x_test_norm, SEQ_LEN, attrs_norm=attrs_norm)
)
nse_attrs = {gid: nse_score(pred_attrs_cfs[:, i], obs_cfs_test[:, i])
             for i, gid in enumerate(loader.gage_ids)}

print("\nNSE — Baseline vs. +Attributes:")
print(f"  {'Gage ID':>10s}  {'Baseline':>10s}  {'+Attrs':>10s}  {'Δ NSE':>8s}")
for gid in loader.gage_ids:
    delta = nse_attrs[gid] - nse_base[gid]
    print(f"  {gid:>10d}  {nse_base[gid]:10.3f}  {nse_attrs[gid]:10.3f}  {delta:+8.3f}")
print(f"  {'Mean':>10s}  {np.nanmean(list(nse_base.values())):10.3f}"
      f"  {np.nanmean(list(nse_attrs.values())):10.3f}"
      f"  {np.nanmean(list(nse_attrs.values())) - np.nanmean(list(nse_base.values())):+8.3f}")


---
## 3. Architecture Modifications: Depth and Convolutional Encoders

Two more structural levers:

### Deeper LSTM

Stacking multiple LSTM layers allows each layer to learn a different level of temporal abstraction — short-term runoff dynamics in layer 1, seasonal soil moisture in layer 2, and so on. Depth is only beneficial when the task genuinely has hierarchical temporal structure and when there's enough data to train the extra parameters.

### Causal convolution front-end

Replacing the linear encoder with 1D **causal convolutions** lets the model explicitly aggregate local temporal context (e.g., a 7-day precipitation event) before the LSTM processes the sequence. "Causal" means each output only depends on *past* inputs — no future leakage.

```
Input (batch, seq_len, features)
 │
 ▼  ConstantPad1d (left-pad, causal)
 ▼  Conv1d -> ReLU     looks at kernel_size consecutive days
 ▼  Conv1d -> ReLU     stacks another level of local context
 │
 ▼  LSTM -> Linear -> output
```


In [ ]:
class DeepLstmModel(nn.Module):
    """Same four-layer pipeline as LstmModel, but with n_layers stacked LSTMs.

    Stacking LSTM layers lets each level learn a different temporal scale —
    layer 1 might capture daily runoff spikes while layer 2 captures slower
    seasonal soil-moisture patterns. Deeper stacks need more dropout to stay
    regularized: the inter-layer dropout is applied automatically by PyTorch
    when n_layers > 1.
    """

    def __init__(self, n_features: int, hidden_size: int = 64,
                 n_layers: int = 2, dropout: float = 0.3):
        super().__init__()
        self.input_proj = nn.Linear(n_features, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, n_layers,
                            dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x       (batch, seq_len, n_features)
        x = torch.relu(self.input_proj(x))
        # x       (batch, seq_len, hidden_size)
        x, _ = self.lstm(x)          # data flows through n_layers LSTM layers
        # x       (batch, seq_len, hidden_size)
        return self.output_proj(self.dropout(x)).squeeze(-1)
        # return  (batch, seq_len)

m1 = DeepLstmModel(N_FEATURES, 64, 2, 0.3).to(device)
print(f"DeepLstmModel       (2-layer LSTM) : {count_params(m1):,} params")

> **Before running the training cells** — look at the `forward` method of each model.
> Where exactly does temporal aggregation happen in `DeepLstmModel` vs. `CausalConvLstmModel`?

In [ ]:
class CausalConvLstmModel(nn.Module):
    """Causal 1D-convolution encoder followed by a single-layer LSTM.

    A Conv1d layer looks at kernel_size consecutive days at once, letting it
    detect short events (e.g. a 7-day storm) before the LSTM processes the
    full sequence. "Causal" means left-padding only — each output day sees
    the current day and past days, never future ones, so there is no leakage.
    """

    def __init__(self, n_features: int, hidden_size: int = 64,
                 kernel_size: int = 7, dropout: float = 0.3):
        super().__init__()
        
        # Left-pad so the conv window never looks into the future
        self.pad  = nn.ConstantPad1d((kernel_size - 1, 0), 0.0)
        self.conv1 = nn.Conv1d(n_features, hidden_size, kernel_size)
        self.pad2  = nn.ConstantPad1d((kernel_size - 1, 0), 0.0)
        self.conv2 = nn.Conv1d(hidden_size, hidden_size, kernel_size)

        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x       (batch, seq_len, n_features)   time-first, from DataLoader
        x = x.permute(0, 2, 1)
        # x       (batch, n_features, seq_len)   Conv1d expects channels-first
        x = torch.relu(self.conv1(self.pad(x)))
        x = torch.relu(self.conv2(self.pad2(x)))
        # x       (batch, hidden_size, seq_len)
        x = x.permute(0, 2, 1)
        # x       (batch, seq_len, hidden_size)  back to time-first for LSTM
        out, _ = self.lstm(x)
        return self.output_proj(self.dropout(out)).squeeze(-1)
        # return  (batch, seq_len)

m2 = CausalConvLstmModel(N_FEATURES, 64, 7, 0.3).to(device)
print(f"CausalConvLstmModel (conv + LSTM)  : {count_params(m2):,} params")

In [ ]:
print("=== Training DeepLstmModel ===")
model_deep = DeepLstmModel(N_FEATURES, hidden_size=64, n_layers=2, dropout=0.3).to(device)

tl_deep, vl_deep = train(
    model=model_deep,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=30,
    verbose=False,
    weights_path=WEIGHTS_DIR / "nb2_deep.pt",
    train_from_scratch=TRAIN_FROM_SCRATCH,
    save_weights=SAVE_WEIGHTS,
)

pred_deep_cfs = denormalize_target(predict_full_timeseries(model_deep, x_test_norm, SEQ_LEN))
nse_deep = {gid: nse_score(pred_deep_cfs[:, i], obs_cfs_test[:, i])
            for i, gid in enumerate(loader.gage_ids)}
print(f"  Mean NSE: {np.nanmean(list(nse_deep.values())):.3f}\n")

**What to look for in the curves above:** if val loss diverges from train loss,
the deeper model is overfitting — try increasing `dropout` before adding more layers.

In [ ]:
print("=== Training CausalConvLstmModel ===")
model_conv = CausalConvLstmModel(N_FEATURES, hidden_size=64, kernel_size=7, dropout=0.3).to(device)

tl_conv, vl_conv = train(
    model=model_conv,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=30,
    verbose=False,
    weights_path=WEIGHTS_DIR / "nb2_conv.pt",
    train_from_scratch=TRAIN_FROM_SCRATCH,
    save_weights=SAVE_WEIGHTS,
)

pred_conv_cfs = denormalize_target(predict_full_timeseries(model_conv, x_test_norm, SEQ_LEN))
nse_conv = {gid: nse_score(pred_conv_cfs[:, i], obs_cfs_test[:, i])
            for i, gid in enumerate(loader.gage_ids)}
print(f"  Mean NSE: {np.nanmean(list(nse_conv.values())):.3f}")

> **Exercise** — Add your own modification.
>
> Ideas to try:
> - **Larger kernel** — change `kernel_size` in `CausalConvLstmModel` to 14 or 30
> - **Residual connection** — add the input back to the LSTM output: `out = lstm_out + input_proj(x)`
> - **Combine attributes + deep** — modify `LstmWithAttrs` to use 2 LSTM layers
> - **Highway LSTM** — add a gating mechanism on the skip connection around the LSTM


In [ ]:
# YOUR TURN — implement one of the modifications above.
# For example: change kernel_size to 14 or 30, re-instantiate, and compare NSE.

# your code here


---
## 4. Model Comparison

| Model | Key feature | Expected benefit |
|-------|-------------|-----------------|
| **Baseline** | hidden=64, 1 layer | Reference point |
| **+Attributes** | Static basin embeddings | Better inter-basin generalization |
| **DeepLSTM** | 2 stacked LSTM layers | Hierarchical temporal features |
| **ConvLSTM** | Causal conv encoder + LSTM | Explicit local pattern extraction |


In [ ]:
all_models = {
    "Baseline": nse_base,
    "+Attributes": nse_attrs,
    "DeepLSTM": nse_deep,
    "ConvLSTM": nse_conv,
}

df_nse = pd.DataFrame(all_models, index=loader.gage_ids)
df_nse.index.name = "Gage ID"
df_nse.loc["Mean"] = df_nse.mean()
df_nse.loc["Median"] = df_nse.iloc[:-1].median()

df_nse.round(3).style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1)


In [ ]:
plot_model_comparison(all_models, loader.gage_ids, save_path="model_comparison.png")


In [ ]:
# Validation loss curves for all variants
lc_data = [
    ("Baseline", vl_base, "#7f8c8d"),
    ("+Attributes", vl_attrs, "#2980b9"),
    ("DeepLSTM", vl_deep, "#27ae60"),
    ("ConvLSTM", vl_conv, "#8e44ad"),
]

plot_validation_loss_comparison(lc_data)



---
## Workshop Summary

Over these two notebooks you have:

1. **Explored** the CAMELS hydrometeorological dataset
2. **Preprocessed** raw time series — temporal splits, normalization, sequence windowing
3. **Built** an LSTM from scratch with `nn.Module`
4. **Trained** with backpropagation, masked loss, and gradient clipping
5. **Evaluated** with Nash-Sutcliffe Efficiency, hydrographs, scatter diagnostics, and flow duration curves
6. **Diagnosed** underfitting, overfitting, and seasonal bias
7. **Engineered features** by adding temporal and seasonal predictors to improve input representation
8. **Augmented** models with static attributes, deeper LSTM structures, and causal convolution encoders


---
### What to explore next

- **More basins** — CAMELS has 671 basins; scale up and see how NSE generalizes
- **More features** — add soil moisture indices or snow water equivalent as forcings
- **Transformer** — replace the LSTM with a multi-head self-attention block
- **Differentiable parameter learning** — combine an LSTM with a physics-based model
- **Uncertainty quantification** — use MC Dropout or ensembles to estimate prediction intervals
